In [1]:
import json, os, re
import numpy as np
import pandas as pd
from collections import defaultdict

docs = [json.loads(l) for l in open("../data/corpus/documents.jsonl")]
texts = [d["text"] for d in docs]

print(f"{len(docs)} documents")
print(pd.Series([d["season"] for d in docs]).value_counts().to_string())
print(pd.Series([d["type"] for d in docs]).value_counts().to_string())

557 documents
2023-24    527
2024-25     30
player_season    497
team_season       60


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
embeddings = np.asarray(embeddings, dtype="float32")
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

os.makedirs("../data/index", exist_ok=True)
np.save("../data/index/embeddings.npy", embeddings)
print(embeddings.shape)

W1102 19:53:29.897000 85294 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

(557, 384)


In [3]:
def search_semantic(query, k=5):
    q = model.encode([query])[0].astype("float32")
    q = q / np.linalg.norm(q)
    scores = embeddings @ q
    top = np.argsort(-scores)[:k]
    return [{**docs[i], "score": float(scores[i])} for i in top]

In [4]:
SEASONS = sorted({d["season"] for d in docs})
PLAYER_SEASONS = sorted({d["season"] for d in docs if d["type"] == "player_season"})
print("All seasons:", SEASONS)
print("Player seasons:", PLAYER_SEASONS)

player_idx = defaultdict(set)
for i, d in enumerate(docs):
    if d["type"] == "player_season":
        full = d["player"].lower()
        player_idx[full].add(i)
        player_idx[full.split()[-1]].add(i)

team_idx = defaultdict(set)
for i, d in enumerate(docs):
    if d["type"] == "team_season":
        team_idx[d["team"].lower()].add(i)
        team_idx[d["team"].lower().split()[-1]].add(i)
        team_idx[d["team_abbr"].lower()].add(i)

print(f"{len(player_idx)} player keys, {len(team_idx)} team keys")

All seasons: ['2023-24', '2024-25']
Player seasons: ['2023-24']
890 player keys, 90 team keys


In [5]:
def parse_query(q):
    ql = q.lower()

    season = None
    m = re.search(r"(20\d\d)[-–](\d\d)", q)
    if m and f"{m.group(1)}-{m.group(2)}" in SEASONS:
        season = f"{m.group(1)}-{m.group(2)}"
    else:
        m = re.search(r"\b(20\d\d)\b", q)
        if m:
            y = m.group(1)
            for s in SEASONS:
                if s.startswith(y) or s.endswith(y[-2:]):
                    season = s
                    break

    players = {n for n in player_idx if len(n) > 3 and n in ql}
    teams   = {n for n in team_idx   if len(n) > 3 and n in ql}
    return {"season": season, "players": players, "teams": teams}

for q in ["How many points did Nikola Jokić average in 2023-24?",
          "What was the Boston Celtics record in 2024-25?",
          "How did Stephen Curry shoot from three?"]:
    print(q, "\n  ", parse_query(q), "\n")

How many points did Nikola Jokić average in 2023-24? 
   {'season': '2023-24', 'players': {'jokić', 'nikola jokić'}, 'teams': set()} 

What was the Boston Celtics record in 2024-25? 
   {'season': '2024-25', 'players': {'boston'}, 'teams': {'boston celtics', 'celtics'}} 

How did Stephen Curry shoot from three? 
   {'season': None, 'players': {'stephen curry', 'curry'}, 'teams': set()} 



In [6]:
def search_hybrid(query, k=5):
    p = parse_query(query)
    candidates = set(range(len(docs)))
    notes = []

    # Narrow by named entities
    entity_hits = set()
    for n in p["players"]:
        entity_hits |= player_idx[n]
    for n in p["teams"]:
        entity_hits |= team_idx[n]
    if entity_hits:
        candidates &= entity_hits

    # Narrow by season — and report honestly if we can't
    season_applied = False
    if p["season"]:
        hits = {i for i, d in enumerate(docs) if d["season"] == p["season"]}
        narrowed = candidates & hits
        if narrowed:
            candidates = narrowed
            season_applied = True
        else:
            notes.append(
                f"No documents match {p['season']} for this entity. "
                f"Player data covers {', '.join(PLAYER_SEASONS)} only."
            )

    if not candidates:
        return {"results": [], "parsed": p, "season_applied": False, "notes": notes}

    q = model.encode([query])[0].astype("float32")
    q = q / np.linalg.norm(q)
    cand = sorted(candidates)
    scores = embeddings[cand] @ q
    order = np.argsort(-scores)[:k]

    return {
        "results": [{**docs[cand[i]], "score": float(scores[i])} for i in order],
        "parsed": p,
        "season_applied": season_applied,
        "notes": notes,
    }

In [7]:
tests = [
    "How many points per game did Nikola Jokić average in 2023-24?",
    "What was the Boston Celtics record in 2023-24?",
    "How did Stephen Curry shoot from three?",
    "What was Anthony Edwards' three-point percentage in 2023-24?",
    "How many points did Luka Dončić average in 2024-25?",
]

for q in tests:
    print(f"\n{'='*72}\nQ: {q}")
    print("\n  SEMANTIC ONLY:")
    for r in search_semantic(q, 3):
        print(f"    [{r['score']:.3f}] {r.get('player') or r.get('team')} — {r['season']}")

    out = search_hybrid(q, 3)
    print(f"\n  HYBRID (season={out['parsed']['season']}, applied={out['season_applied']}):")
    for r in out["results"]:
        print(f"    [{r['score']:.3f}] {r.get('player') or r.get('team')} — {r['season']}")
    for n in out["notes"]:
        print(f"    NOTE: {n}")


Q: How many points per game did Nikola Jokić average in 2023-24?

  SEMANTIC ONLY:
    [0.802] Nikola Jokić — 2023-24
    [0.722] Nikola Jović — 2023-24
    [0.680] Jusuf Nurkić — 2023-24

  HYBRID (season=2023-24, applied=True):
    [0.802] Nikola Jokić — 2023-24

Q: What was the Boston Celtics record in 2023-24?

  SEMANTIC ONLY:
    [0.736] Boston Celtics — 2023-24
    [0.723] Boston Celtics — 2024-25
    [0.589] Philadelphia 76ers — 2023-24

  HYBRID (season=2023-24, applied=True):
    [0.736] Boston Celtics — 2023-24
    [0.530] Brandon Boston — 2023-24

Q: How did Stephen Curry shoot from three?

  SEMANTIC ONLY:
    [0.532] Stephen Curry — 2023-24
    [0.466] Seth Curry — 2023-24
    [0.389] Jae'Sean Tate — 2023-24

  HYBRID (season=None, applied=False):
    [0.532] Stephen Curry — 2023-24
    [0.466] Seth Curry — 2023-24

Q: What was Anthony Edwards' three-point percentage in 2023-24?

  SEMANTIC ONLY:
    [0.637] Anthony Edwards — 2023-24
    [0.562] Anthony Gill — 2023-24
  

In [8]:
import pickle
with open("../data/index/docs.pkl", "wb") as f:
    pickle.dump(docs, f)
print("Saved")

Saved
